In [ ]:
# ============================================================
# TXT -> SBML -> libSBML validation -> COBRApy -> FBA
# Designed for the custom metabolic-model TXT format
# ============================================================

# If needed, uncomment this first:
# %pip install cobra python-libsbml

from pathlib import Path
import re
import math
import warnings

import libsbml
import cobra
from cobra import Model, Reaction, Metabolite


# ============================================================
# 0. SETTINGS
# ============================================================

# CHANGE THIS ONLY IF YOUR TXT HAS A DIFFERENT NAME/LOCATION
TXT_FILE = Path("1752-0509-4-31-S2.TXT")

SBML_FILE = Path("1752-0509-4-31-S2_cobra.xml")

# The biomass reaction in your TXT
OBJECTIVE_RXN = "R_BIOMASS"

# Expected exchange from your original TXT
EXPECTED_EXCHANGES = {
    "fru[e]": (-1000.0, 0.0),
    "cellulose[e]": (-1000.0, 0.0),
    "nh4[e]": (-1000.0, 0.0),
    "pi[e]": (-1000.0, 0.0),
}


# ============================================================
# 1. FIND TXT FILE
# ============================================================

if not TXT_FILE.exists():
    candidates = list(Path(".").glob("*.TXT")) + list(Path(".").glob("*.txt"))

    if len(candidates) == 1:
        TXT_FILE = candidates[0]
    else:
        raise FileNotFoundError(
            f"Could not find {TXT_FILE}.\n"
            f"TXT files found: {candidates}"
        )

print("=" * 70)
print("SOURCE FILE")
print("=" * 70)
print(TXT_FILE.resolve())


# ============================================================
# 2. PARSE THE ORIGINAL TXT
# ============================================================

lines = TXT_FILE.read_text(errors="replace").splitlines()

exchanges = {}
gprs = {}
raw_reactions = []

section = None

for line in lines:

    s = line.strip()

    if s == "#exchanges:":
        section = "exchanges"
        continue

    if s == "#gprs:":
        section = "gprs"
        continue

    if s == "#model:":
        section = "model"
        continue

    if not s:
        continue

    # -------------------------
    # Exchanges
    # -------------------------
    if section == "exchanges":

        parts = line.split()

        if len(parts) >= 3:
            try:
                met = parts[0]
                lb = float(parts[1])
                ub = float(parts[2])
                exchanges[met] = (lb, ub)
            except ValueError:
                pass

    # -------------------------
    # GPRs
    # -------------------------
    elif section == "gprs":

        parts = line.split("\t")

        if len(parts) >= 3 and parts[0].strip() == "rg":
            rid = parts[1].strip()
            gpr = parts[2].strip()

            if gpr != ".":
                gprs[rid] = gpr

    # -------------------------
    # Reactions
    # -------------------------
    elif section == "model":

        parts = line.split("\t")

        if len(parts) >= 6:
            rid = parts[0].strip()
            name = parts[1].strip()
            gene = parts[2].strip()
            pathway = parts[3].strip()
            ec = parts[4].strip()
            equation = parts[5].strip()

            raw_reactions.append({
                "id": rid,
                "name": name,
                "gene": gene,
                "pathway": pathway,
                "ec": ec,
                "equation": equation,
            })


print("\n" + "=" * 70)
print("TXT PARSING")
print("=" * 70)

print(f"Exchange entries : {len(exchanges)}")
print(f"GPR entries      : {len(gprs)}")
print(f"Reactions        : {len(raw_reactions)}")


# ============================================================
# 3. SANITY CHECKS AGAINST ORIGINAL TXT
# ============================================================

assert len(exchanges) == 54, (
    f"Expected 54 exchange entries, found {len(exchanges)}"
)

reaction_ids = {r["id"] for r in raw_reactions}

for rid in [
    "R_BIOMASS",
    "R_BIOMASScell",
    "R_CELLOME",
]:
    assert rid in reaction_ids, f"{rid} missing from TXT"

for met, expected_bounds in EXPECTED_EXCHANGES.items():

    assert met in exchanges, (
        f"{met} missing from TXT exchange section"
    )

    assert exchanges[met] == expected_bounds, (
        f"{met}: expected {expected_bounds}, "
        f"found {exchanges[met]}"
    )

print("✓ 54 exchange entries found")
print("✓ fructose exchange = (-1000, 0)")
print("✓ cellulose exchange = (-1000, 0)")
print("✓ NH4 exchange = (-1000, 0)")
print("✓ phosphate exchange = (-1000, 0)")
print("✓ biomass reactions found")


# ============================================================
# 4. EQUATION PARSER
# ============================================================

def parse_metabolite(raw):

    raw = raw.strip()

    match = re.fullmatch(
        r"(.+?)\[([^\]]+)\]",
        raw
    )

    if match:
        return match.group(1), match.group(2)

    return raw, None


def split_equation(equation):

    equation = equation.strip()

    default_compartment = None

    # Example:
    # [c] : ATP + H2O --> ADP + Pi
    match = re.match(
        r"^\[([^\]]+)\]\s*:\s*(.*)$",
        equation
    )

    if match:
        default_compartment = match.group(1)
        equation = match.group(2).strip()

    if "<==>" in equation:
        lhs, rhs = equation.split("<==>", 1)
        reversible = True

    elif "<=>" in equation:
        lhs, rhs = equation.split("<=>", 1)
        reversible = True

    elif "-->" in equation:
        lhs, rhs = equation.split("-->", 1)
        reversible = False

    else:
        raise ValueError(
            f"Cannot determine reaction direction:\n{equation}"
        )

    return (
        default_compartment,
        lhs.strip(),
        rhs.strip(),
        reversible
    )


def parse_side(side, default_compartment):

    terms = []

    if not side:
        return terms

    # IMPORTANT:
    # Split on + only when it is a separator.
    # This model uses normal metabolite names, so this is sufficient.
    for term in side.split("+"):

        term = term.strip()

        if not term:
            continue

        # coefficient + metabolite
        match = re.match(
            r"^\s*"
            r"(?:(\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?)\s+)?"
            r"(.+?)"
            r"\s*$",
            term
        )

        if not match:
            raise ValueError(
                f"Could not parse reaction term: {term}"
            )

        coefficient = float(match.group(1) or 1.0)

        raw_met = match.group(2).strip()

        met, compartment = parse_metabolite(raw_met)

        if compartment is None:
            compartment = default_compartment

        if compartment is None:
            raise ValueError(
                f"No compartment specified for metabolite "
                f"'{raw_met}'"
            )

        terms.append(
            (met, compartment, coefficient)
        )

    return terms


# ============================================================
# 5. PARSE ALL REACTIONS
# ============================================================

parsed_reactions = []

for r in raw_reactions:

    default, lhs, rhs, reversible = split_equation(
        r["equation"]
    )

    reactants = parse_side(
        lhs,
        default
    )

    products = parse_side(
        rhs,
        default
    )

    parsed_reactions.append({
        **r,
        "reactants": reactants,
        "products": products,
        "reversible": reversible,
        "default_compartment": default,
    })


# ============================================================
# 6. CREATE COBRA MODEL DIRECTLY
# ============================================================

model = Model("TXT_converted_model")

# Avoid excessive warnings during reconstruction
warnings.filterwarnings(
    "ignore",
    category=UserWarning
)


# ------------------------------------------------------------
# Metabolites
# ------------------------------------------------------------

metabolite_objects = {}

for r in parsed_reactions:

    for met, comp, coeff in (
        r["reactants"] + r["products"]
    ):

        key = (met, comp)

        if key not in metabolite_objects:

            # COBRApy IDs
            cobra_id = f"{met}_{comp}"

            m = Metabolite(
                cobra_id,
                name=met,
                compartment=comp
            )

            metabolite_objects[key] = m


# Add metabolites
model.add_metabolites(
    list(metabolite_objects.values())
)


# ------------------------------------------------------------
# Reactions
# ------------------------------------------------------------

cobra_reactions = {}

for r in parsed_reactions:

    rxn = Reaction(r["id"])

    rxn.name = (
        "" if r["name"] == "."
        else r["name"]
    )

    # Direction from TXT
    if r["reversible"]:

        rxn.lower_bound = -1000
        rxn.upper_bound = 1000

    else:

        rxn.lower_bound = 0
        rxn.upper_bound = 1000

    # Stoichiometry
    stoich = {}

    for met, comp, coeff in r["reactants"]:

        m = metabolite_objects[(met, comp)]

        stoich[m] = stoich.get(m, 0) - coeff

    for met, comp, coeff in r["products"]:

        m = metabolite_objects[(met, comp)]

        stoich[m] = stoich.get(m, 0) + coeff

    rxn.add_metabolites(stoich)

    # EC annotation
    if r["ec"] and r["ec"] != ".":

        rxn.annotation["ec-code"] = r["ec"]

    # Store source pathway
    if r["pathway"] and r["pathway"] != ".":

        rxn.notes["pathway"] = r["pathway"]

    model.add_reactions([rxn])

    cobra_reactions[r["id"]] = rxn


# ============================================================
# 7. CREATE EXCHANGE REACTIONS
# ============================================================

for raw_met, (lb, ub) in exchanges.items():

    met, comp = parse_metabolite(raw_met)

    if comp is None:
        raise ValueError(
            f"Exchange metabolite lacks compartment: {raw_met}"
        )

    key = (met, comp)

    if key not in metabolite_objects:

        m = Metabolite(
            f"{met}_{comp}",
            name=met,
            compartment=comp
        )

        metabolite_objects[key] = m
        model.add_metabolites([m])

    m = metabolite_objects[key]

    # EX_fructose -> EX_fru_e
    exchange_id = f"EX_{met}_{comp}"

    if exchange_id in model.reactions:
        rxn = model.reactions.get_by_id(exchange_id)

    else:

        rxn = Reaction(exchange_id)

        rxn.name = f"Exchange {met}"

        # Exchange convention:
        # negative = uptake
        # positive = secretion
        rxn.add_metabolites({
            m: -1
        })

        model.add_reactions([rxn])

    rxn.lower_bound = lb
    rxn.upper_bound = ub


# ============================================================
# 8. GPRs
# ============================================================

for rid, gpr in gprs.items():

    if rid not in model.reactions:
        continue

    rxn = model.reactions.get_by_id(rid)

    try:
        rxn.gene_reaction_rule = gpr

    except Exception as e:

        print(
            f"WARNING: could not assign GPR to {rid}: {e}"
        )


# ============================================================
# 9. CHECK CRITICAL REACTIONS
# ============================================================

print("\n" + "=" * 70)
print("CRITICAL MODEL CHECKS")
print("=" * 70)

for rid in [
    "R_BIOMASS",
    "R_BIOMASScell",
    "R_CELLOME",
]:

    rxn = model.reactions.get_by_id(rid)

    print(
        f"{rid:15s}",
        "bounds =", rxn.bounds
    )

    print(
        "   ",
        rxn.reaction
    )


# ============================================================
# 10. CHECK FRUCTOSE
# ============================================================

fru_ex = model.reactions.get_by_id(
    "EX_fru_e"
)

print("\nFructose exchange:")
print(
    fru_ex.id,
    fru_ex.bounds,
    fru_ex.reaction
)

assert fru_ex.bounds == (-1000.0, 0.0)

print("✓ fructose exchange correctly preserved")


# ============================================================
# 11. OBJECTIVE
# ============================================================

assert OBJECTIVE_RXN in model.reactions

model.objective = OBJECTIVE_RXN

print("\nObjective:")
print(model.objective)


# ============================================================
# 12. SAVE COBRA SBML
# ============================================================

cobra.io.write_sbml_model(
    model,
    SBML_FILE
)

print("\nSBML written:")
print(SBML_FILE.resolve())


# ============================================================
# 13. LIBSBML VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("LIBSBML VALIDATION")
print("=" * 70)

doc = libsbml.readSBML(
    str(SBML_FILE)
)

print(
    "SBML errors reported:",
    doc.getNumErrors()
)

if doc.getNumErrors():

    for i in range(doc.getNumErrors()):

        error = doc.getError(i)

        print(
            f"\n[{i}] severity={error.getSeverity()}"
        )

        print(
            error.getMessage()
        )

else:

    print("✓ No libSBML errors")


# ============================================================
# 14. RELOAD SBML WITH COBRAPY
# ============================================================

print("\n" + "=" * 70)
print("RELOADING SBML WITH COBRAPY")
print("=" * 70)

model2 = cobra.io.read_sbml_model(
    SBML_FILE
)

print(
    "Reactions :",
    len(model2.reactions)
)

print(
    "Metabolites:",
    len(model2.metabolites)
)

print(
    "Genes:",
    len(model2.genes)
)


# ============================================================
# 15. CRITICAL POST-SBML CHECKS
# ============================================================

print("\n" + "=" * 70)
print("POST-SBML CHECKS")
print("=" * 70)

for rid in [
    "R_BIOMASS",
    "R_BIOMASScell",
    "R_CELLOME",
    "EX_fru_e",
]:

    assert rid in model2.reactions

    rxn = model2.reactions.get_by_id(rid)

    print(
        f"{rid:15s}",
        rxn.bounds,
        "|",
        rxn.reaction
    )


# Fructose MUST still be -1000 / 0
assert model2.reactions.get_by_id(
    "EX_fru_e"
).bounds == (-1000.0, 0.0)

print("\n✓ EX_fru_e survived SBML conversion correctly")


# ============================================================
# 16. CHECK BIOMASS STOICHIOMETRY
# ============================================================

biomass = model2.reactions.get_by_id(
    "R_BIOMASS"
)

print("\nBiomass:")
print(biomass.reaction)

assert (
    "cellmass_c" in biomass.metabolites
    or
    any(
        m.name == "cellmass"
        for m in biomass.metabolites
    )
)

print("✓ biomass reaction contains cellmass")


# ============================================================
# 17. CHECK CELLOME
# ============================================================

cellome = model2.reactions.get_by_id(
    "R_CELLOME"
)

print("\nCELLOME:")
print(cellome.reaction)

print(
    "CELLOME bounds:",
    cellome.bounds
)